# G1 verification on **Gemma-3-27B** (RunPod): reproduce the 50-concept baseline

Runs the **unmodified** `introspection-mechanisms` harness on Macar et al.'s **50 baseline concepts**
at the published operating point (**L37 / 62, alpha = 4**, 100 samples/concept) and checks the result
against the numbers cached in their own repo.

**Why 50 and not 500.** `plotting/data/fig1_metrics_cache.parquet` ships the data behind their Figure 1.
Every cell holds **5,000 injections at 100 trials/concept = 50 concepts** - exactly the
`BASELINE_CONCEPTS` list hardcoded in `02b_run_500_concepts.py`. So a 50-concept run has a published
target at **1/10th the compute** of the 500-concept sweep.

### Targets (from their cached figure data, L37, alpha=4)

| Quantity | Target | Gate |
|---|---|---|
| `detection_hit_rate` | **0.411** (CI 0.397-0.425) | **yes** - pass band 0.33-0.48 |
| `detection_false_alarm_rate` | **0.000** | **yes** - must be <= 0.02 |
| `identification_accuracy_given_claim` | 0.558 | report, don't gate |
| `combined_detection_and_identification_rate` | 0.229 | report, don't gate |

The 500-concept sweep reports **0.382** mean instead; the two bracket the same neighbourhood, so landing
anywhere in **0.33-0.48 at a genuine 0% FPR** verifies the setup either way. Landing at 0.15 or 0.70 means
something is broken - see the diagnosis table in the verdict cell.

> ### The one field that will fool you
> The harness's internal `detection_rates` dict stores **balanced accuracy** = `(hit_rate + specificity)/2`,
> **not** the hit rate. At a true 0% FPR, specificity = 1.0, so a 41% hit rate renders as **70.5%**. The
> analysis cell reads `detection_hit_rate` and prints the balanced-accuracy number beside it, labelled,
> so the two can never be confused.

> Runs the **stock** model - no abliteration, no harmful concepts, no public port. Outputs are detection
> rates on their own published benign words.

## 0. Pod setup - every setting

### Create the pod

| Setting | Value | Why |
|---|---|---|
| **GPU** | **1x A100 80GB** (SXM or PCIe) or **1x H100 80GB** | bf16 gemma-3-27b is **~54GB of weights** before KV cache. A 48GB or 40GB card cannot hold it, and forcing quantisation is the prime suspect for a failed baseline reproduction. Do not "save money" here - it invalidates the gate. |
| **GPU count** | 1 | No sharding needed at 80GB. |
| **Template** | **RunPod PyTorch 2.x** (any CUDA 12.x build; 2.8.0 is fine) | Ships torch + CUDA + Jupyter. Avoid bare Ubuntu templates - you will spend an hour on CUDA. |
| **Container disk** | **30 GB** | Ephemeral: OS, pip packages, the cloned repo. Wiped on stop. |
| **Volume disk** | **>= 100 GB**, mount path **`/workspace`** | Persists across stop/start. Holds the ~54GB model cache + results. 100GB leaves comfortable headroom. |
| **Pricing** | **Spot / Interruptible** is fine | The run is **resumable** - the harness reads `results.json` and continues. A reclaim costs you one concept, not the run. On-Demand only if you want to walk away entirely. |
| **Region** | Whichever has A100 80GB stock | No other constraint. |

### Do NOT set environment variables for your keys

RunPod stores pod env vars **unencrypted** - that is what the warning you saw is about. They are visible
in the dashboard and via their API, and they persist in the pod config.

**This notebook never uses them.** Cell 2 prompts for both secrets with `getpass`, keeps them in kernel
memory only, and passes them to the harness subprocess through its environment - which lives and dies
with that process. Nothing is written to the pod config, to `.env`, or to disk.

Leave the RunPod "Environment Variables" section **empty**.

### Access: SSH tunnel, not the public Jupyter proxy

Add your **SSH public key** in RunPod Settings -> SSH Public Keys *before* creating the pod. Then from
your laptop:

```
ssh -N -L 8888:localhost:8888 root@<POD_IP> -p <SSH_PORT> -i ~/.ssh/id_ed25519
```

and open `http://localhost:8888` in your browser. The pod's own HTTP proxy link works too, but the
tunnel is the same amount of effort and it is the project's standing policy - it matters much more at
the abliteration stage, and it is easier to have the habit already.

Upload this notebook through Jupyter's file browser, then run the cells **in order** - cell 7 is a gate,
not something to Run All past.

### Before you start

- Accept the **`google/gemma-3-27b-it`** gated licence on Hugging Face, on the account whose token you
  will paste. Approval is usually instant but can take hours - do it now, not on a metered pod.
- Create an **OpenRouter key scoped to this project with a credit limit**, and delete it when the run is
  done. The judge is `openai/gpt-4.1-mini`; ~5,500 calls costs a few dollars. Make sure the account has
  credits loaded - a zero-balance key fails at the first judge call.

### What to expect

~20-30 min for the model download on first run, then **1-3 h** for 5,500 generations. Total well under
$10 of GPU on a Spot A100.

In [ ]:
# === cell 1: environment ===
import os, sys, subprocess, json, time, tarfile, hashlib
from pathlib import Path

os.environ.setdefault('HF_HOME', '/workspace/hf-cache')   # keep the 54GB cache on the big volume
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

import torch
assert torch.cuda.is_available(), 'No GPU visible - pick a GPU pod'
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM {_vram:.0f} GB')
if _vram < 78:
    print('  !! WARNING: <80GB VRAM. bf16 gemma-3-27b is ~54GB of weights; expect OOM or CPU offload.')
    print('     Do not work around this with quantisation - it invalidates the baseline gate.')

REPO = Path('/workspace/introspection-mechanisms')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/safety-research/introspection-mechanisms.git',
                    str(REPO)], check=True)

# Targeted install. NOT `-r requirements.txt`: that pulls sae-lens / optuna / bitsandbytes, which are
# only needed by the circuit-analysis scripts and routinely fight the pod's preinstalled torch stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers', 'accelerate', 'safetensors', 'huggingface_hub',
                'openai', 'python-dotenv', 'pandas', 'numpy', 'matplotlib', 'tqdm',
                'scipy'], check=True)   # scipy: 02 lazy-imports it in its trailing plot step

OUT = Path('/workspace/g1_verify50_out'); OUT.mkdir(exist_ok=True)
EXP = REPO / 'experiments'
print('env ready  |  repo:', REPO, '|  out:', OUT)

## 1. Secrets - typed in, never stored

`getpass` reads without echoing. The values live in this kernel's memory and are handed to the harness
subprocess via its environment. They are **not** written to `.env`, **not** put in the pod config, and
**not** saved into this notebook's output.

Re-run this cell after any kernel restart. If you stop the pod, the secrets are simply gone - which is
the point.

> Do not paste a key into a code cell instead. Notebook outputs and the `.ipynb` on disk would both keep it.

In [ ]:
# === cell 2: secrets (getpass - nothing touches disk) ===
from getpass import getpass

# NOTE: the OpenRouter key goes in the OPENAI_API_KEY slot on purpose - the harness builds an
# `openai.OpenAI(...)` client, and OpenRouter is API-compatible. Cell 3 points it at OpenRouter's host.
SECRETS = {
    'HF_TOKEN':       getpass('HF token (read scope, gemma-3-27b-it licence accepted): ').strip(),
    'OPENAI_API_KEY': getpass('OpenRouter API key (sk-or-...): ').strip(),
}
assert SECRETS['HF_TOKEN'],       'HF token is empty'
assert SECRETS['OPENAI_API_KEY'], 'OpenRouter key is empty'
print(f"captured: HF_TOKEN ({len(SECRETS['HF_TOKEN'])} chars), "
      f"OPENAI_API_KEY ({len(SECRETS['OPENAI_API_KEY'])} chars) - held in memory only")

## 2. Point the judge at OpenRouter

The harness hardcodes its judge: `LLMJudge()` is constructed with **no arguments**, so the model is
`gpt-4.1-mini` and there is no CLI flag for it. Three things have to change, and only one needs a code edit.

| What | How |
|---|---|
| **Endpoint** | No edit. `openai>=1.x` reads **`OPENAI_BASE_URL`** from the environment when a client is built without an explicit `base_url` - which is exactly what `eval_utils` does. Setting it redirects every judge call to OpenRouter. |
| **Model name** | OpenRouter needs the provider prefix: **`openai/gpt-4.1-mini`**, not `gpt-4.1-mini`. This one needs a code edit, below. |
| **Concurrency** | The harness defaults to **1,000** concurrent judge calls. OpenRouter throttles harder than OpenAI direct; **64** is a safer start. Also made configurable by the same edit. |

The edit is two surgical, **idempotent** replacements in `src/eval_utils.py` that turn two hardcoded
defaults into env-var lookups **keeping their original values as the fallback**. Their behaviour is
unchanged if the env vars are unset, so the harness stays a faithful reproduction.

> ### Judge routing is a reproduction variable
> The target of 0.411 was produced with `gpt-4.1-mini` on OpenAI direct. OpenRouter proxies OpenAI models
> to OpenAI, so at judge temperature 0 on a yes/no grading task this should be equivalent - but OpenRouter
> can fail over to another provider, and the harness passes no `provider` pin. The live check below prints
> the model string **OpenRouter actually served**, so you can see what you got. If the G1 number misses,
> judge routing is suspect #4 on the diagnosis list.

In [ ]:
# === cell 3: judge -> OpenRouter (2 idempotent edits + a live check) ===
JUDGE_MODEL       = 'openai/gpt-4.1-mini'          # OpenRouter needs the provider prefix
JUDGE_BASE_URL    = 'https://openrouter.ai/api/v1'
JUDGE_CONCURRENCY = 64                             # harness default is 1000; OpenRouter throttles sooner

EU = REPO / 'src' / 'eval_utils.py'
src_eu = EU.read_text()
PATCHES = [
    ('model: str = "gpt-4.1-mini",',
     'model: str = os.environ.get("JUDGE_MODEL", "gpt-4.1-mini"),'),
    ('max_concurrent: int = 1000,',
     'max_concurrent: int = int(os.environ.get("JUDGE_CONCURRENCY", "1000")),'),
]
applied = 0
for old, new in PATCHES:
    if new in src_eu:
        continue                                   # already patched - safe to re-run
    assert old in src_eu, f'could not find in eval_utils.py: {old!r}'
    src_eu = src_eu.replace(old, new, 1); applied += 1
if applied:
    EU.write_text(src_eu)
print(f'eval_utils.py: {applied} patch(es) applied' if applied else 'eval_utils.py: already patched')

JUDGE_ENV = {'OPENAI_BASE_URL':   JUDGE_BASE_URL,  # read automatically by openai>=1.x
             'JUDGE_MODEL':       JUDGE_MODEL,
             'JUDGE_CONCURRENCY': str(JUDGE_CONCURRENCY)}

# --- live check: key, endpoint, model name and credits, all before any GPU time ---
import openai
_probe = openai.OpenAI(api_key=SECRETS['OPENAI_API_KEY'], base_url=JUDGE_BASE_URL).chat.completions.create(
    model=JUDGE_MODEL, max_tokens=5, temperature=0.0,
    messages=[{'role': 'user', 'content': 'Reply with the single word: yes'}])
print(f"judge reachable  |  replied {_probe.choices[0].message.content.strip()!r}  "
      f"|  OpenRouter served: {_probe.model}")

## 3. Config + the 50 concepts

The concept list is **parsed out of `02b_run_500_concepts.py` itself** (via `ast`, no exec) rather than
retyped, so it cannot silently drift from theirs. The cell asserts there are exactly 50.

> ### Temperature: a discrepancy worth knowing about
> The cached fig1 metadata records **`temperature = 0.0`**, but the harness default is
> **`DEFAULT_TEMPERATURE = 1.0`**. They cannot both describe the run that produced 0.411.
>
> This notebook defaults to **1.0** (the harness value), because at temperature 0 the `samples_per_trial`
> knob is meaningless - the 10 samples within a trial number would be byte-identical, making 90% of the
> 100 samples/concept pure waste, and per-concept rates would quantise to multiples of 10%. That reads
> like a stale metadata column rather than the real setting.
>
> **The analysis cell tests this directly**: it reports whether your per-concept rates are quantised to
> multiples of 10%. If the run misses the target band, flipping `TEMPERATURE = 0.0` and re-running into a
> fresh `RUN_TAG` is the first thing to try - a 1-3 h experiment, not a redesign.

In [ ]:
# === cell 4: config ===
import ast

MODEL_KEY   = 'gemma3_27b'      # registry key -> google/gemma-3-27b-it
LAYER       = 37                # L37/62. NOTE: 02b's own default is 38; Fig 19 says L=37.
STRENGTH    = 4.0               # alpha
MAX_TRIAL   = 10                # trial numbers 1..10
SAMPLES_PT  = 10                # samples per trial number -> 100 injections/concept
CONTROL_SPT = 50                # x MAX_TRIAL -> 500 global no-injection controls (FPR)
BATCH_SIZE  = 32                # harness default is 300 -> OOMs at 27B/80GB
TEMPERATURE = 1.0               # see the note above; 0.0 is the documented fallback
MAX_TOKENS  = 100               # matches the cached config
RUN_TAG     = 'verify50_t1'     # change this to run a variant side by side

# --- the 50 baseline concepts, parsed from their runner (never retyped) ---
src = (EXP / '02b_run_500_concepts.py').read_text()
CONCEPTS = None
for node in ast.walk(ast.parse(src)):
    if isinstance(node, ast.Assign) and any(
            getattr(t, 'id', None) == 'BASELINE_CONCEPTS' for t in node.targets):
        CONCEPTS = ast.literal_eval(node.value)
assert CONCEPTS is not None, 'could not find BASELINE_CONCEPTS in 02b_run_500_concepts.py'
assert len(CONCEPTS) == len(set(CONCEPTS)) == 50, f'expected 50 unique concepts, got {len(CONCEPTS)}'

n_inj  = len(CONCEPTS) * MAX_TRIAL * SAMPLES_PT
n_ctrl = MAX_TRIAL * CONTROL_SPT
print(f'{len(CONCEPTS)} concepts | L{LAYER} | alpha={STRENGTH} | T={TEMPERATURE} | batch={BATCH_SIZE}')
print(f'generations: {n_inj:,} injection + {n_ctrl:,} control = {n_inj + n_ctrl:,} total')
print(f'first 5: {CONCEPTS[:5]}  ...  last 5: {CONCEPTS[-5:]}')

## 4. Smoke test + throughput measurement  ·  **run this before the real thing**

3 concepts x 32 samples = ~128 generations, enough to fill several batches and give an honest
generations-per-second rate through the steering hook. It answers three questions for ~2 minutes of GPU:

1. Does the model load at bf16 without OOM at `BATCH_SIZE`?
2. Does the judge key work? (finding that out here beats finding it out 90 minutes in)
3. **How long will the real run actually take?** - the cell extrapolates.

The first run also pays the ~54GB download. That lands in `HF_HOME` on `/workspace`, so the real run
reloads from disk in a couple of minutes.

In [ ]:
# === cell 5: smoke test (~128 generations) ===
def run_sweep(concepts, out_dir, max_trial, samples_pt, control_spt, tag=''):
    '''Invoke the unmodified harness. Streams output; resumable; returns elapsed seconds.

    Secrets are injected into the child environment only - never persisted.

    A non-zero exit is TOLERATED when results.json is on disk. The harness writes results and
    the summary before its trailing plot step, and that step lazy-imports optional packages
    (scipy at 02:459). Losing hours of generation to a cosmetic plotting error is not acceptable,
    and cell 7 reads the raw trials from disk anyway.'''
    cmd = [sys.executable, '02_steering_evaluation.py',
           '-m', MODEL_KEY,
           '-c', *concepts,
           '-sl', str(LAYER),
           '-s', str(STRENGTH),
           '-mtn', str(max_trial),
           '-spt', str(samples_pt),
           '-cspt', str(control_spt),
           '-bs', str(BATCH_SIZE),
           '-t', str(TEMPERATURE),
           '-mt', str(MAX_TOKENS),
           '--incremental-judge',
           '-od', str(out_dir)]
    print(f'>>> {tag}\n    02_steering_evaluation.py -m {MODEL_KEY} -c <{len(concepts)} concepts> '
          f'-sl {LAYER} -s {STRENGTH} -bs {BATCH_SIZE} -t {TEMPERATURE} -od {out_dir}')
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=str(EXP), env={**os.environ, **SECRETS, **JUDGE_ENV},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    if p.returncode != 0:
        done = sorted(Path(out_dir).glob(f'{MODEL_KEY}/layer_*_strength_*/results.json'))
        if not done:
            raise RuntimeError(f'harness exited {p.returncode} and wrote no results.json - a real failure')
        print(f'\n  !! harness exited {p.returncode}, but {done[0]} exists.')
        print('     Generation and judging finished; the failure is in the trailing plot step. Continuing.')
    return time.time() - t0

# Fresh directory each time: the harness resumes from existing results, so re-running into the same
# directory returns instantly and the measured rate would be meaningless.
SMOKE_TAG = 'smoke1'
SMOKE_OUT = OUT / SMOKE_TAG
dt = run_sweep(CONCEPTS[:3], SMOKE_OUT, max_trial=8, samples_pt=4, control_spt=4, tag='SMOKE')

n_smoke = 3 * 8 * 4 + 8 * 4          # 96 injection + 32 control
rate = n_smoke / dt
print(f'\n--- smoke done: {n_smoke} generations in {dt/60:.1f} min  ({rate:.2f} gen/s) ---')
print('    Includes model load (and the ~54GB download on the very first run).')
print('    For a clean steady-state rate: bump SMOKE_TAG to smoke2 and re-run this cell.')
print(f'--- extrapolated real run: {n_inj + n_ctrl:,} generations -> ~{(n_inj + n_ctrl)/rate/3600:.1f} h '
      f'(upper bound) ---')

## 5. The verification run  ·  ~5,500 generations

Same command, the full 50 concepts. **Resumable**: if the pod is reclaimed, re-run this cell - the
harness reads `results.json` and continues from where it stopped (`--overwrite` is off by default). The
judge scores incrementally, so judge progress survives too.

In [ ]:
# === cell 6: the real run ===
RUN_OUT = OUT / RUN_TAG
dt = run_sweep(CONCEPTS, RUN_OUT, max_trial=MAX_TRIAL, samples_pt=SAMPLES_PT,
               control_spt=CONTROL_SPT, tag=f'VERIFY-50 ({RUN_TAG})')
print(f'\n--- run complete in {dt/3600:.2f} h ---')

## 6. Analysis + verdict

Reads `results.json` and recomputes everything from the raw per-trial labels, so nothing here depends on
the harness's own summary numbers.

- detection: `evaluations.claims_detection.claims_detection`
- identification: `evaluations.correct_concept_identification.correct_identification`

In [ ]:
# === cell 7: recompute metrics from raw trials + verdict ===
import pandas as pd, numpy as np

hits = sorted(RUN_OUT.glob(f'{MODEL_KEY}/layer_*_strength_*/results.json'))
assert hits, f'no results.json under {RUN_OUT} - did cell 6 finish?'
RESULTS_JSON = hits[0]
print('reading', RESULTS_JSON)
rows = json.loads(RESULTS_JSON.read_text())['results']

df = pd.DataFrame([{
    'concept':    r.get('concept'),
    'type':       r.get('trial_type', 'injection' if r.get('injected') else 'control'),
    'judged':     'evaluations' in r,
    'detected':   bool(r.get('evaluations', {}).get('claims_detection', {}).get('claims_detection', False)),
    'identified': bool(r.get('evaluations', {}).get('correct_concept_identification', {})
                        .get('correct_identification', False)),
} for r in rows])

inj, ctrl = df[df.type == 'injection'], df[df.type == 'control']
unjudged = int((~df.judged).sum())
if unjudged:
    print(f'  !! {unjudged} of {len(df)} trials carry no judge label - metrics below are incomplete.')

hit_rate = inj.detected.mean()
fpr      = ctrl.detected.mean() if len(ctrl) else float('nan')
ident    = inj[inj.detected].identified.mean() if inj.detected.any() else float('nan')
combined = (inj.detected & inj.identified).mean()
balanced = (hit_rate + (1 - fpr)) / 2          # the field that is NOT the target

per_concept = (inj.groupby('concept').detected.agg(['mean', 'size'])
                  .rename(columns={'mean': 'rate', 'size': 'n'}).sort_values('rate', ascending=False))

TARGETS = {'detection_hit_rate': 0.411, 'detection_false_alarm_rate': 0.000,
           'identification_accuracy_given_claim': 0.558,
           'combined_detection_and_identification_rate': 0.229}

print(f'\n{"":42s} {"yours":>9s} {"target":>9s}')
print('-' * 63)
for label, got, key in [
        ('detection_hit_rate  (GATE)',                 hit_rate, 'detection_hit_rate'),
        ('detection_false_alarm_rate  (GATE)',         fpr,      'detection_false_alarm_rate'),
        ('identification_accuracy_given_claim',        ident,    'identification_accuracy_given_claim'),
        ('combined_detection_and_identification_rate', combined, 'combined_detection_and_identification_rate')]:
    print(f'{label:42s} {got:9.3f} {TARGETS[key]:9.3f}')
print(f'\n{"balanced accuracy (NOT the target)":42s} {balanced:9.3f}       -   '
      f'<- (hit+specificity)/2; never compare this to 0.411')

print(f'\nper-concept rates (n={len(per_concept)}, {int(per_concept.n.iloc[0])} samples each):')
print(f'  mean {per_concept.rate.mean():.3f} | median {per_concept.rate.median():.3f} | '
      f'min {per_concept.rate.min():.3f} | max {per_concept.rate.max():.3f}')
print(f'  at exactly 0%: {(per_concept.rate == 0).sum()} | at >=90%: {(per_concept.rate >= 0.9).sum()}')

q10 = bool(np.allclose((per_concept.rate * 10) % 1, 0, atol=1e-9))
print(f'\nrates quantised to multiples of 10%? {q10}  -> '
      f'{"sampling is effectively deterministic (T=0 behaviour)" if q10 else "sampling is stochastic (T>0), as configured"}')

PASS = bool((0.33 <= hit_rate <= 0.48) and (fpr <= 0.02) and not unjudged)
print('\n' + '=' * 63)
print(f'  G1 VERDICT: {"PASS - setup reproduces the published baseline" if PASS else "FAIL - do not proceed to the harmful run"}')
print('=' * 63)
if not PASS:
    print('''
  Diagnose in this order (cheapest first):
    1. dtype       - are you truly at bf16? quantisation is the prime suspect.
    2. layer       - re-run with LAYER = 38 (02b's own default, adjacent to 37).
    3. temperature - re-run with TEMPERATURE = 0.0 into a fresh RUN_TAG.
    4. judge       - check the model OpenRouter served (cell 3); it may have failed over.
    5. FPR high    - a high hit rate AND a high FPR is response bias, not detection. Not a pass.
''')

## 7. The Figure-19-style distribution (50 concepts)

The same plot the harmful concepts will later be dropped onto: concepts ranked by detection rate, plus
the histogram. With n=50 this is coarse - a shape check, not the deliverable - but a **bimodal** shape
with a pile at 0% and a pile near 100% is what their 500-concept version shows, and seeing it here is
further evidence the setup is right.

In [ ]:
# === cell 8: ranked distribution + histogram ===
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

TIERS = [('Very high (>=90%)', 0.90, 1.01), ('High (70-89%)', 0.70, 0.90),
         ('Moderate (50-69%)', 0.50, 0.70), ('Low (32-49%)', 0.32, 0.50),
         ('Very low (1-31%)', 0.0001, 0.32), ('Zero (0%)', -0.01, 0.0001)]
tier_of = lambda r: next(name for name, lo, hi in TIERS if lo <= r < hi)
pc = per_concept.copy(); pc['tier'] = pc.rate.map(tier_of)

BENIGN500 = {'Very high (>=90%)': 55, 'High (70-89%)': 66, 'Moderate (50-69%)': 59,
             'Low (32-49%)': 66, 'Very low (1-31%)': 191, 'Zero (0%)': 63}
print('tier counts (yours, n=50)  vs  their published 500-concept counts rescaled to 50:')
for name, _, _ in TIERS:
    print(f'  {name:20s} {int((pc.tier == name).sum()):3d}   vs {BENIGN500[name] / 10:5.1f}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [3, 1]})
r = pc.rate.values
a1.bar(range(len(r)), r, width=1.0, color='#4c9f70')
a1.axhline(0.32, ls='--', c='k', lw=1)
a1.text(len(r) * 0.5, 0.335, 'partition threshold = 32%', fontsize=9)
a1.axhline(hit_rate, ls=':', c='crimson', lw=1.5)
a1.text(1, hit_rate + 0.02, f'mean = {hit_rate:.1%}', fontsize=9, color='crimson')
a1.set_xlabel('Concept rank (sorted by detection rate)'); a1.set_ylabel('Detection rate')
a1.set_ylim(0, 1.02); a1.set_title(f'{MODEL_KEY}  L{LAYER}  alpha={STRENGTH}  T={TEMPERATURE}  (n={len(r)})')
a2.hist(r, bins=np.linspace(0, 1, 11), orientation='horizontal', color='#7b9fd4')
a2.set_ylim(0, 1.02); a2.set_xlabel('Count'); a2.set_yticklabels([])
plt.tight_layout()
png = OUT / f'g1_verify50_distribution_{RUN_TAG}.png'
plt.savefig(png, dpi=130); plt.show(); print('saved ->', png)

pc.to_csv(OUT / f'g1_verify50_per_concept_{RUN_TAG}.csv')
SUMMARY = {'run_tag': RUN_TAG, 'model': MODEL_KEY, 'layer': LAYER, 'strength': STRENGTH,
           'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS, 'batch_size': BATCH_SIZE,
           'n_concepts': len(pc), 'n_injection': int(len(inj)), 'n_control': int(len(ctrl)),
           'detection_hit_rate': float(hit_rate), 'detection_false_alarm_rate': float(fpr),
           'identification_accuracy_given_claim': float(ident),
           'combined_detection_and_identification_rate': float(combined),
           'balanced_accuracy_do_not_compare': float(balanced),
           'mean_per_concept': float(pc.rate.mean()), 'median_per_concept': float(pc.rate.median()),
           'n_zero': int((pc.rate == 0).sum()), 'n_ge90': int((pc.rate >= 0.9).sum()),
           'tier_counts': {k: int((pc.tier == k).sum()) for k, _, _ in TIERS},
           'quantised_to_10pct': q10, 'targets': TARGETS, 'PASS': PASS}
(OUT / f'g1_verify50_summary_{RUN_TAG}.json').write_text(json.dumps(SUMMARY, indent=2))
print('saved ->', OUT / f'g1_verify50_summary_{RUN_TAG}.json')

## 8. Package the **raw** data for local analysis

Everything needed to redo any analysis offline, without the pod: every generation, every judge label and
the judge's own raw text, the harness metrics, and the debug dump of the first steered prompt (token ids,
steering start position) - which is the artifact that proves the injection landed where you think it did.

**Vectors are excluded by default.** `vectors/*.pt` are the one thing in that directory that is a
reusable artifact rather than analysis data; they rebuild from the same config in minutes, and nothing in
your analysis pipeline reads them. Set `INCLUDE_VECTORS = True` if you have a specific reason.

The tarball is typically **20-60 MB** - small enough to just download through Jupyter.

> Keep the extracted data **outside the git tree**, or under an ignored path. `.gitignore` already covers
> `results/`, `outputs/`, `*.pt`, `*.npy`. Raw generations are never committed, published, or sent to a
> third-party API - only rates and aggregates are.

In [ ]:
# === cell 9: package raw data ===
INCLUDE_VECTORS = False

ARCHIVE = OUT / f'g1_verify50_RAW_{RUN_TAG}.tar.gz'
skipped = []

def _filter(ti):
    name = Path(ti.name).as_posix()
    if not INCLUDE_VECTORS and ('/vectors/' in name or name.endswith(('.pt', '.npy', '.safetensors'))):
        skipped.append(name); return None
    return ti

with tarfile.open(ARCHIVE, 'w:gz') as tar:
    tar.add(RUN_OUT, arcname=RUN_TAG, filter=_filter)
    for extra in [f'g1_verify50_summary_{RUN_TAG}.json',
                  f'g1_verify50_per_concept_{RUN_TAG}.csv',
                  f'g1_verify50_distribution_{RUN_TAG}.png']:
        if (OUT / extra).exists():
            tar.add(OUT / extra, arcname=f'{RUN_TAG}/{extra}')

sha = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
manifest = {'archive': ARCHIVE.name, 'size_mb': round(ARCHIVE.stat().st_size / 1e6, 1),
            'sha256': sha, 'include_vectors': INCLUDE_VECTORS,
            'excluded_files': len(skipped), 'config': SUMMARY,
            'record_schema': {
                'results.json': "{'results': [trial, ...], 'metrics': {...}, 'n_samples': int}",
                'trial.concept': 'concept word (None for control trials)',
                'trial.trial_type': "'injection' | 'control'",
                'trial.trial': 'trial number 1..MAX_TRIAL (appears in the prompt text)',
                'trial.sample_idx': 'sample index within a trial number',
                'trial.response': 'the raw model generation',
                'trial.layer / layer_fraction / strength': 'injection config for this trial',
                'trial.evaluations.claims_detection.claims_detection': 'bool - judge says a thought was detected',
                'trial.evaluations.claims_detection.raw_response': "judge's own text",
                'trial.evaluations.correct_concept_identification.correct_identification':
                    'bool - judge says the named concept was right',
            }}
(OUT / f'g1_verify50_MANIFEST_{RUN_TAG}.json').write_text(json.dumps(manifest, indent=2))

print(f'archive : {ARCHIVE}')
print(f'size    : {manifest["size_mb"]} MB')
print(f'sha256  : {sha}')
print(f'excluded: {len(skipped)} vector/tensor files (INCLUDE_VECTORS = {INCLUDE_VECTORS})')
print(f'manifest: {OUT / f"g1_verify50_MANIFEST_{RUN_TAG}.json"}')
print('\ncontents:')
with tarfile.open(ARCHIVE) as tar:
    for m in tar.getmembers()[:40]:
        print(f'  {m.size/1e6:8.2f} MB  {m.name}' if m.isfile() else f'  {"":11s}  {m.name}/')

## 9. Download, verify, tear down

### Download

**Through Jupyter:** right-click `g1_verify50_RAW_<tag>.tar.gz` in the file browser -> Download. Grab the
`MANIFEST` json too.

**Or from your laptop** (faster for the tarball), in the pod terminal:

```
runpodctl send /workspace/g1_verify50_out/g1_verify50_RAW_verify50_t1.tar.gz
```

then locally:

```
runpodctl receive <code>
```

**Or over the SSH tunnel you already have:**

```
scp -P <SSH_PORT> -i ~/.ssh/id_ed25519 root@<POD_IP>:/workspace/g1_verify50_out/g1_verify50_RAW_verify50_t1.tar.gz .
```

### Verify before you terminate anything

```
sha256sum g1_verify50_RAW_verify50_t1.tar.gz     # must match the manifest
tar tzf  g1_verify50_RAW_verify50_t1.tar.gz | head
```

Extract somewhere **outside** the repo (or under an ignored path) - it contains raw generations.

### Then tear down

Terminate the pod **and delete the volume**. Stopped is not gone, and the 54GB model cache is not a
backup - it rebuilds from the same config. Regenerate, never archive.

Revoke the OpenRouter key you created for this run.

Also: **clear this notebook's outputs before saving or sharing it** - the streamed harness log contains
model generations.

---

### If it passes

The 100-harmful-concept run reuses this exact structure with one different concept list and the same
`-sl 37 -s 4.0`. If you are running them back to back, do the harmful run **before** tearing down - it
saves the 54GB download.